# AgenTekki — Tax Agent Filler (Remake)

Over the past weeks you've built tools that automatically extract tasks from meetings, generate articles, and manage inventories. Now it's time for the **fully agentic** project: an AI agent that autonomously files a Zurich tax return.

## What's different about this notebook

Previous approaches dumped **all** taxpayer documents into every LLM call. This notebook uses a smarter **Plan-then-Execute** architecture:

1. **Plan phase** — The LLM sees only document *names and descriptions* (not content). It decides which documents it actually needs and outlines a strategy.
2. **Execute phase** — The LLM receives only the *selected* documents and produces the field mappings.

This is closer to how real AI agents work: **perceive → plan → retrieve → act**.

You will also use a **local LLM** via OLLAMA instead of cloud APIs.

## Learning Objectives

By the end of this notebook you will:
- Understand the **Plan-Execute** pattern in AI agents
- Build and use an **LLM service abstraction** with a provider pattern
- Implement **selective document retrieval** (the LLM chooses what to read)
- **Visualize** agent decision-making (plans, document selection, reasoning)
- Wire a **full autonomous pipeline**: scan → plan → retrieve → fill → navigate → submit

## 1. Setup

In [ ]:
#@title Clone Repository & Install Dependencies (click to run) { display-mode: "form" }

import os, sys
os.chdir("/content")

!rm -rf /content/project3-agentic-tax-filler
!git clone -b remake-adrian https://github.com/eth-bmai-fs26/project3-agentic-tax-filler.git /content/project3-agentic-tax-filler

# Purge cached modules if re-running
for mod_name in list(sys.modules):
    if "mcp_server" in mod_name:
        del sys.modules[mod_name]

sys.path.insert(0, "/content/project3-agentic-tax-filler")

!pip install -q openai

print("Done!")

In [3]:
#@title Build & Serve the React Frontend (click to run) { display-mode: "form" }

import subprocess, os, threading, http.server, functools, socket

FRONTEND_DIR = "/content/project3-agentic-tax-filler"
BUILD_DIR = os.path.join(FRONTEND_DIR, "EnvDemoV1", "dist")

# The frontend lives on the 'frontend' branch — fetch and checkout
os.chdir(FRONTEND_DIR)
subprocess.run(["git", "fetch", "origin", "frontend"], check=True)
subprocess.run(["git", "checkout", "frontend"], check=True)

# Install and build
os.chdir(os.path.join(FRONTEND_DIR, "EnvDemoV1"))
subprocess.run(["npm", "install"], check=True)
subprocess.run(["npm", "run", "build"], check=True)
os.chdir("/content")

# Find a free port
def _find_free_port(start=3000, end=3020):
    for port in range(start, end):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(("", port))
                return port
            except OSError:
                continue
    raise RuntimeError("No free port found")

PORT = _find_free_port()
handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=BUILD_DIR)
httpd = http.server.HTTPServer(("", PORT), handler)
threading.Thread(target=httpd.serve_forever, daemon=True).start()
print(f"Frontend served at http://localhost:{PORT}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/project3-agentic-tax-filler'

In [1]:
# Initialize the LLM Service
from mcp_server.llm_service import LLMService

# ============================================================
# STUDENT TODO: Create an LLM service instance.
#
# Option A — Local OLLAMA (recommended):
#   llm = LLMService.ollama(model="llama3.1:8b")
#
# Option B — ETH proxy:
#   llm = LLMService.openai_compatible(
#       model="openai/gpt-4o-mini",
#       base_url="https://litellm.sph-prod.ethz.ch/v1",
#       api_key="YOUR_KEY_HERE",
#   )
#
# Option C — Gemini (free tier):
#   llm = LLMService.openai_compatible(
#       model="gemini-2.0-flash-lite",
#       base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
#       api_key="YOUR_KEY_HERE",
#   )
# ============================================================

llm = LLMService.ollama(model="qwen3:4b")  # <-- FILL THIS IN

# Quick test
response = llm.chat(
    system="You are a helpful assistant.",
    user="Say 'hello' in German. Reply with just the word.",
    max_tokens=20,
)
print(f"LLM says: {response}")

LLM says: Okay, the user wants me to say 'hello' in German and just reply with the word.


## 2. Simulation — Explore the Tax Portal

Before building an agent, understand what we're automating. The cell below renders the tax portal UI. Click around and explore!

**Try this:**
- Navigate through the sections using the sidebar (Personal, Income, Deductions, Wealth)
- Click "Next" to move between sub-pages
- Add rows in the Children or Bank Accounts sections
- Check the Review page to see how all data comes together
- Notice the field types: text inputs, number inputs, dropdowns, checkboxes

In [2]:
#@title Explore the Tax Portal (click around, nothing is saved!) { display-mode: "form" }

from IPython.display import display, HTML
import os

BUILD_DIR = "/content/project3-agentic-tax-filler/EnvDemoV1/dist"
with open(os.path.join(BUILD_DIR, "index.html")) as f:
    html_content = f.read()

html_content = html_content.replace('"\/assets/', f'"http://localhost:{PORT}/assets/')
html_content = html_content.replace('"./assets/', f'"http://localhost:{PORT}/assets/')
html_content = html_content.replace('"/assets/', f'"http://localhost:{PORT}/assets/')

display(HTML(html_content))

<>:10: SyntaxWarning: invalid escape sequence '\/'
<>:10: SyntaxWarning: invalid escape sequence '\/'
/var/folders/2m/05xs488d04vf6j0yp4lr7_100000gn/T/ipykernel_53318/4250835723.py:10: SyntaxWarning: invalid escape sequence '\/'
  html_content = html_content.replace('"\/assets/', f'"http://localhost:{PORT}/assets/')


FileNotFoundError: [Errno 2] No such file or directory: '/content/project3-agentic-tax-filler/EnvDemoV1/dist/index.html'

**Reflection** (discuss with your neighbor):
- How many pages does the form have? How many sub-pages?
- What kinds of fields appear? (text, number, dropdown, checkbox, date)
- If you were an AI agent, which documents would you need for the Income page? For Deductions?
- What information can you get from the field *labels* alone, without reading any documents?

## 3. Plan Discovery — Think Before You Act

An agent needs a **plan** before it can act. Let's think about what steps are needed to fill one page of the tax form.

The agent follows a cycle: **Perceive** (scan the page) → **Plan** (decide what docs to read and how to fill) → **Act** (read docs, fill fields).

Below, you'll write plans *by hand* — the same kind of plan the LLM will later generate automatically.

In [ ]:
# Here's what a plan for the Personal page looks like:
personal_plan = {
    "page": "personal",
    "reasoning": "Personal info (name, address, DOB, AHV) comes from profile.json. "
                 "Occupation and employer come from the Lohnausweis.",
    "documents_needed": ["profile.json", "lohnausweis.txt"],
    "guides_needed": [],
    "strategy": "Extract name, address, DOB, AHV from profile.json. "
                "Get occupation and employer from Lohnausweis header.",
}

# ============================================================
# STUDENT TODO: Write a plan for the Income page.
# Think: which documents contain salary, interest, dividends?
# ============================================================
income_plan = {
    "page": "income",
    "reasoning": ...,           # <-- FILL IN: why do you need those documents?
    "documents_needed": ...,    # <-- FILL IN: list of filenames
    "guides_needed": ...,       # <-- FILL IN: any guides needed? (can be [])
    "strategy": ...,            # <-- FILL IN: how will you extract the values?
}

# ============================================================
# STUDENT TODO: Write a plan for the Deductions page.
# Think: meal deduction, commuting, pillar 3a, donations...
# ============================================================
deductions_plan = {
    "page": "deductions",
    "reasoning": ...,
    "documents_needed": ...,
    "guides_needed": ...,
    "strategy": ...,
}

# Print your plans
import json
for plan in [personal_plan, income_plan, deductions_plan]:
    print(f"\n=== {plan['page'].upper()} ===")
    print(f"  Reasoning: {plan['reasoning']}")
    print(f"  Documents: {plan['documents_needed']}")
    print(f"  Strategy:  {plan['strategy']}")

**Key insight:** You just did exactly what the agent's **Plan phase** will do! The difference: the LLM will write these plans automatically, seeing only document *names and descriptions* — not their content. This is how the agent decides what information to retrieve before making expensive document reads.

## 4. Available Tools & Documents

Let's see what the agent has to work with.

In [ ]:
# Initialize the MCP Server with MockBridge (no real browser needed yet)
from mcp_server.server import MCPServer
from mcp_server.bridges.mock import MockBridge
from mcp_server.doc_catalog import build_document_catalog

REPO = "/content/project3-agentic-tax-filler"
PERSONA = "anna_meier"

mock_server = MCPServer(
    persona_folder=f"{REPO}/personas/{PERSONA}",
    guides_folder=f"{REPO}/guides",
    bridge=MockBridge(),
)

# Show the document catalog
catalog = build_document_catalog(mock_server)

from IPython.display import display, HTML
rows = "".join(
    f"<tr><td><code>{d['filename']}</code></td><td>{d['description']}</td></tr>"
    for d in catalog
)
display(HTML(f'''
<h3>Documents available for {PERSONA}</h3>
<table style="border-collapse:collapse; width:100%;">
<tr style="background:#4a90d9; color:white;">
    <th style="padding:8px; text-align:left;">Filename</th>
    <th style="padding:8px; text-align:left;">Description</th>
</tr>
{rows}
</table>
<p><b>Note:</b> The Plan phase sees only this table — not the actual file contents.</p>
'''))

In [ ]:
# Show the 9 available tools
tools_info = [
    ("scan_page()", "Returns all visible fields, buttons, and text on the current form page"),
    ("fill_field(locator, value)", "Enters a value into a form field identified by its locator"),
    ("click_element(locator)", "Clicks a button or navigation link"),
    ("submit_form()", "Submits the completed tax return"),
    ("list_documents()", "Lists all files in the persona's document folder"),
    ("read_document(filepath)", "Reads and extracts content from a document (.txt, .csv, .pdf, .json)"),
    ("list_guides()", "Returns available tax guide topics"),
    ("fetch_guide(url)", "Returns the full text of a specific tax guide"),
    ("ask_user(question)", "Asks the simulated taxpayer a question (penalized if unnecessary)"),
]

rows = "".join(
    f"<tr><td><code>server.{name}</code></td><td>{desc}</td></tr>"
    for name, desc in tools_info
)
display(HTML(f'''
<h3>Available Tools</h3>
<table style="border-collapse:collapse; width:100%;">
<tr style="background:#2d7d2d; color:white;">
    <th style="padding:8px; text-align:left;">Tool</th>
    <th style="padding:8px; text-align:left;">Description</th>
</tr>
{rows}
</table>
'''))

In [ ]:
# What does scan_page() actually return? Let's look at the income page.
from mcp_server.bridges.mock import MockBridge

mock = MockBridge()
mock.frontend.current_page = "income"
page_data = mock.scan_page()

import json
print(f"Page: {page_data['page_name']}")
print(f"Number of elements: {len(page_data['elements'])}\n")

for el in page_data["elements"]:
    if el.get("type") in ("button", "text"):
        continue
    print(f"  {el.get('type'):10s}  {el.get('locator', ''):50s}  label: {el.get('label', '')}")

print("\nThis is what the LLM sees when deciding which fields to fill.")

## 5. Think Step — Single Page Reasoning

Now we implement the **two-phase reasoning** for a single page:

```
scan_page()  →  page fields
                    │
        ┌───────────┴───────────┐
        │     PLAN PHASE        │
        │  fields + doc catalog │──→  plan JSON
        │  (no doc content!)    │     (which docs to read,
        └───────────────────────┘      strategy, reasoning)
                    │
        ┌───────────┴───────────┐
        │   SELECTIVE RETRIEVAL │
        │  read only the docs   │──→  doc contents
        │  the plan requested   │
        └───────────────────────┘
                    │
        ┌───────────┴───────────┐
        │    EXECUTE PHASE      │
        │  plan + doc contents  │──→  {locator: value} JSON
        │  + fields             │
        └───────────────────────┘
                    │
              fill_field() for each
```

In [ ]:
# ── PLAN PHASE PROMPT ──────────────────────────────────────────

PLAN_SYSTEM_PROMPT = """You are an expert Swiss tax accountant AI.
You are planning how to fill one page of a Zurich tax return.

You will see:
1. The fields on the current form page (locators, labels, types)
2. A catalog of available documents (names and descriptions only -- NOT their content)
3. A list of available tax guides (titles only)
4. What you have already filled on previous pages

Your job: decide which documents and guides you need to read, and outline your strategy for filling the fields.

## Response Format (JSON only, no other text)
{
  "_reasoning": "Your step-by-step reasoning about what this page needs",
  "documents_needed": ["filename1.txt", "filename2.csv"],
  "guides_needed": ["guide-name"],
  "strategy": "Brief description of how you will fill each field",
  "fields_to_fill": ["field-locator-1", "field-locator-2"],
  "ask_user_needed": false
}

## Rules
- Only request documents that are relevant to the current page's fields.
- If no fields need filling (e.g. attachments, review), return an empty documents_needed list.
- Be specific in your strategy: mention which document contains which value.
- Do NOT guess document content -- you can only see names and descriptions.
"""


def build_plan_user_message(page_name, fields, catalog, guides_list, filled_history):
    """Build the user message for the PLAN phase."""
    # Filter to fillable fields only
    fillable = [f for f in fields if f.get("type") not in ("button", "text")]
    fields_desc = json.dumps(fillable, indent=2)

    catalog_text = "\n".join(
        f"- {d['filename']}: {d['description']}" for d in catalog
    )
    guides_text = "\n".join(f"- {g}" for g in guides_list) if guides_list else "(no guides available)"

    history_lines = []
    for prev_page, prev_fields in filled_history.items():
        entries = ", ".join(f"{k}={v}" for k, v in prev_fields.items() if not k.startswith("_"))
        if entries:
            history_lines.append(f"  {prev_page}: {entries}")
    history_text = "\n".join(history_lines) if history_lines else "  (none yet)"

    # ============================================================
    # STUDENT TODO: Assemble the user message.
    # Combine the variables above into a clear message with sections:
    #   - Current Page + Fields
    #   - Document Catalog
    #   - Available Guides
    #   - Previously Filled
    # ============================================================
    user_msg = ...  # <-- FILL THIS IN

    return user_msg

In [ ]:
# ── EXECUTE PHASE PROMPT ───────────────────────────────────────
import json, re, time
from pathlib import Path

EXECUTE_SYSTEM_PROMPT = """You are an expert Swiss tax accountant AI filling a Zurich tax return.

You will see:
1. Your own plan (reasoning and strategy from the planning phase)
2. The actual content of the documents you requested
3. The fields on the current form page
4. Tax guides with rules and thresholds
5. What you already filled on previous pages

Your task: fill the fields on this page using the document content and your plan.

## Response Format (JSON only, no other text)
Return a JSON object mapping field locators to values:
{"_thought": "reasoning here", "field-locator-1": "value1", "field-locator-2": "value2"}

## Rules
- ONLY use locators shown in the current page fields -- NEVER guess locators.
- If a field already has the correct value, do NOT include it.
- If a field is required but you have no data, skip it (don't make up values).
- If no fields apply, return {}.
- Do NOT fill fields with empty strings, "None", "0", or placeholder text.
- Include a "_thought" key explaining your reasoning.
"""

# Section-specific prompts (appended to EXECUTE_SYSTEM_PROMPT per page)
SECTION_PROMPTS = {
    "personal": """## Personal Details
- Name, Address, DOB, AHV: Extract from profile.json.
- Parse address into street, streetNumber, zip, city.
- Marital status: "ledig", "verheiratet", "geschieden", etc.
- Occupation & Employer: From the Lohnausweis header.
- Children: Fill BOTH name AND dateOfBirth if applicable.
""",

    "income": """## Income Section
- Bruttolohn = Lohnausweis field 1 (gross salary).
- AHV/IV/EO = Lohnausweis field 9.
- BVG = Lohnausweis field 10.
- Dienstaltersgeschenke (field 6) are ALREADY INCLUDED in Bruttolohn -- do NOT add separately.
- Interest income: sum ALL "Zinsabschluss" entries from bank statement.
- Self-employment: only fill if freelance documents exist.
""",

    "deductions": """## Deductions Section
- Verpflegung (meals): Lohnausweis field 14.1 Ja = CHF 1,600; Nein = CHF 3,200.
- Berufsauslagen Pauschale: 3% of Nettolohn (field 12), min CHF 2,000, max CHF 4,000.
- Fahrkosten (commuting): use ZVV receipt amount.
- Pillar 3a: max CHF 7,056 (employed with BVG).
- Donations: min CHF 100, look for "Spende" in bank statement.
- Schuldzinsen: sum ALL mortgage interest from bank statement.
""",

    "wealth": """## Wealth Section
- Bank balance: LAST line of bank statement CSV (Dec 31 balance).
- Bank interest: sum ALL "Zinsabschluss" entries.
- Securities: market value as of Dec 31.
- Real estate: use Steuerwert from Gemeinde assessment, NOT market value.
""",

    "attachments": "## Attachments\nFile uploads cannot be filled programmatically. Return {}.",
    "review": "## Review\nThis is the final review page. Return {}.",
}


def build_execute_user_message(page_name, fields, plan, doc_contents, guide_contents, filled_history):
    """Build the user message for the EXECUTE phase."""
    fillable = [f for f in fields if f.get("type") not in ("button", "text")]
    fields_desc = json.dumps(fillable, indent=2)

    docs_text = "\n\n".join(
        f"=== {name} ===\n{content[:8000]}"
        for name, content in doc_contents.items()
    )
    guides_text = "\n\n".join(
        f"=== {name} ===\n{content[:3000]}"
        for name, content in guide_contents.items()
    )

    history_lines = []
    for prev_page, prev_fields in filled_history.items():
        entries = ", ".join(f"{k}={v}" for k, v in prev_fields.items() if not k.startswith("_"))
        if entries:
            history_lines.append(f"  {prev_page}: {entries}")
    history_text = "\n".join(history_lines) if history_lines else "  (none yet)"

    plan_text = json.dumps(plan, indent=2) if isinstance(plan, dict) else str(plan)

    # ============================================================
    # STUDENT TODO: Assemble the user message.
    # Combine the variables above into a clear message with sections:
    #   - Your Plan (from the planning phase)
    #   - Document Contents (the actual data)
    #   - Tax Guides (if any)
    #   - Previously Filled
    #   - Current Page + Fields
    # ============================================================
    user_msg = ...  # <-- FILL THIS IN

    return user_msg

In [ ]:
# ── THINK_STEP: Two-phase reasoning for a single page ─────────

def think_step(server, llm, page_name, elements, catalog, all_guides, filled_history, section_prompts):
    """Plan-then-Execute reasoning for one page.

    Phase 1 (Plan):   LLM sees field labels + doc catalog -> outputs a plan
    Phase 2 (Execute): LLM sees plan + selected doc contents -> outputs field mappings

    Returns
    -------
    tuple of (plan_dict, mapping_dict)
        plan_dict:    the plan JSON from phase 1
        mapping_dict: {locator: value} from phase 2
    """
    fields = [e for e in elements if e.get("type") not in ("button", "text")]
    if not fields:
        return {}, {}

    guides_list = list(all_guides.keys())

    # ════════════════════════════════════════════════════════════
    # PHASE 1: PLAN
    # ════════════════════════════════════════════════════════════
    # STUDENT TODO: Call the LLM to create a plan.
    # Use: llm.chat_json(), PLAN_SYSTEM_PROMPT, build_plan_user_message()
    plan = ...  # <-- FILL THIS IN

    print(f"  Plan: need docs {plan.get('documents_needed', [])}")
    print(f"  Strategy: {plan.get('strategy', 'none')}")

    # ════════════════════════════════════════════════════════════
    # SELECTIVE RETRIEVAL
    # ════════════════════════════════════════════════════════════
    # STUDENT TODO: Read only the documents the plan requested.
    doc_contents = {}
    for doc_name in plan.get("documents_needed", []):
        ...  # <-- FILL IN: call server.read_document(), store content

    # STUDENT TODO: Fetch only the guides the plan requested.
    guide_contents = {}
    for guide_name in plan.get("guides_needed", []):
        ...  # <-- FILL IN: look up in all_guides dict

    # ════════════════════════════════════════════════════════════
    # PHASE 2: EXECUTE
    # ════════════════════════════════════════════════════════════
    section = page_name.split("/")[0]
    section_prompt = section_prompts.get(section, "")
    system_msg = EXECUTE_SYSTEM_PROMPT + "\n" + section_prompt

    # STUDENT TODO: Call the LLM to get field mappings.
    # Use: llm.chat_json(), system_msg, build_execute_user_message()
    mapping = ...  # <-- FILL THIS IN

    return plan, mapping

In [ ]:
# ── Test think_step on the Income page (MockBridge) ───────────

from mcp_server.bridges.mock import MockBridge

test_bridge = MockBridge()
test_bridge.frontend.current_page = "income"

test_server = MCPServer(
    persona_folder=f"{REPO}/personas/{PERSONA}",
    guides_folder=f"{REPO}/guides",
    bridge=test_bridge,
)

test_catalog = build_document_catalog(test_server)

# Load guides (may be empty if guides/ folder doesn't exist yet)
def load_all_guides(guides_folder):
    guides = {}
    folder = Path(guides_folder)
    if not folder.exists():
        return guides
    for f in sorted(folder.iterdir()):
        if f.suffix in (".html", ".txt", ".md"):
            guides[f.stem] = f.read_text(encoding="utf-8")
    return guides

all_guides = load_all_guides(f"{REPO}/guides")

page_data = test_server.scan_page()
print(f"Testing on page: {page_data['page_name']}")
print(f"Fillable fields: {len([e for e in page_data['elements'] if e.get('type') not in ('button', 'text')])}\n")

plan, mapping = think_step(
    server=test_server,
    llm=llm,
    page_name=page_data["page_name"],
    elements=page_data["elements"],
    catalog=test_catalog,
    all_guides=all_guides,
    filled_history={},
    section_prompts=SECTION_PROMPTS,
)

print("\nField mappings from LLM:")
for k, v in mapping.items():
    if not k.startswith("_"):
        print(f"  {k} = {v}")
    elif k == "_thought":
        print(f"  Reasoning: {v}")

**Checkpoint:** If `think_step` returned reasonable values for the income page (e.g. gross salary around 142,000 for Anna Meier), you're ready to visualize the plan and build the full loop.

If the values look wrong, check:
- Is `build_plan_user_message` sending the document catalog correctly?
- Is `build_execute_user_message` including the actual document content?
- Is the plan requesting the right documents?

## 6. Plan Visualization

One advantage of the Plan-Execute split: we can **visualize** the agent's decision-making before it acts. This makes the agent's reasoning transparent and debuggable.

In [ ]:
# Pre-built visualization function
from IPython.display import display, HTML

def visualize_plan(page_name, plan, doc_catalog):
    """Render the agent's plan as a styled HTML card."""
    selected = set(plan.get("documents_needed", []))

    docs_html = ""
    for doc in doc_catalog:
        is_selected = doc["filename"] in selected
        icon = "&#9745;" if is_selected else "&#9744;"
        color = "#2d7d2d" if is_selected else "#999"
        weight = "bold" if is_selected else "normal"
        docs_html += (
            f'<div style="color:{color}; font-weight:{weight}; margin:2px 0;">'
            f'{icon} <code>{doc["filename"]}</code> — {doc["description"]}</div>'
        )

    fields_html = ""
    for f in plan.get("fields_to_fill", []):
        fields_html += f'<code style="background:#e8f5e9; padding:2px 6px; margin:2px; border-radius:3px;">{f}</code> '

    reasoning = plan.get("_reasoning", "No reasoning provided")
    strategy = plan.get("strategy", "No strategy provided")

    html = f"""
    <div style="border:2px solid #4a90d9; border-radius:10px; padding:16px; margin:12px 0;
                background:linear-gradient(135deg, #f8faff 0%, #eef3ff 100%); font-family:sans-serif;">
        <h3 style="color:#4a90d9; margin-top:0;">Plan: {page_name}</h3>
        <div style="background:white; border-radius:6px; padding:12px; margin:8px 0;">
            <b>Reasoning:</b>
            <blockquote style="border-left:3px solid #4a90d9; padding-left:12px; color:#333; margin:8px 0;">
                {reasoning}
            </blockquote>
        </div>
        <div style="background:white; border-radius:6px; padding:12px; margin:8px 0;">
            <b>Documents selected:</b><br/>
            {docs_html}
        </div>
        <div style="background:white; border-radius:6px; padding:12px; margin:8px 0;">
            <b>Strategy:</b> {strategy}
        </div>
        <div style="background:white; border-radius:6px; padding:12px; margin:8px 0;">
            <b>Fields to fill:</b><br/>{fields_html if fields_html else "<i>not specified</i>"}
        </div>
    </div>
    """
    display(HTML(html))


# Demo: visualize the plan from our test
if plan:
    visualize_plan(page_data["page_name"], plan, test_catalog)
else:
    print("No plan to visualize yet. Run the think_step test cell first.")

## 7. Think Loop — Full Pipeline

Now we wire `think_step` into the full navigation loop. The code drives page-by-page navigation; your `think_step` handles the reasoning on each page.

The loop: **scan** → **plan** → **retrieve** → **execute** → **fill** → **next page** → repeat until review → **submit**.

In [ ]:
# ── Helper: add dynamic rows ──────────────────────────────────

def add_rows_if_needed(server, elements, num_rows_needed=0):
    """Click 'Add Row' buttons to create dynamic table rows."""
    add_btns = [e for e in elements
                if e.get("type") == "button"
                and "add-row" in (e.get("locator") or "").lower()]
    if not add_btns:
        return None

    existing_rows = set()
    for e in elements:
        loc = e.get("locator") or ""
        match = re.search(r'-(\d+)-', loc)
        if match:
            existing_rows.add(int(match.group(1)))

    rows_to_add = max(0, (num_rows_needed or 1) - len(existing_rows))
    if rows_to_add > 0:
        for _ in range(rows_to_add):
            for btn in add_btns:
                loc = btn.get("locator")
                if loc:
                    print(f"  Adding row: {loc}")
                    server.click_element(loc)
        return server.scan_page()
    return None


# ── Helper: fill fields from mapping ─────────────────────────

def fill_page(server, mapping, page_name, filled_history):
    """Execute fill_field for each locator:value in the mapping."""
    filled = {}
    thought = mapping.pop("_thought", None)
    if thought:
        print(f"  Reasoning: {thought}")

    for locator, value in mapping.items():
        if locator.startswith("_"):
            continue
        result = server.fill_field(locator, str(value))
        if result.get("success"):
            filled[locator] = value
            print(f"  Filled: {locator} = {value}")
        else:
            print(f"  FAILED: {locator}: {result.get('error', 'unknown')}")

    if filled:
        filled_history[page_name] = filled
    return filled

In [ ]:
# ── THE FULL THINK LOOP ───────────────────────────────────────

def think_loop(server, llm, max_steps=100):
    """Full agent: Plan-Execute on every page with visualization."""

    catalog = build_document_catalog(server)
    all_guides = load_all_guides(f"{REPO}/guides")
    filled_history = {}
    plans = []
    total_fields = 0

    # Auto-navigate away from root/overview
    initial = server.scan_page()
    page_name = initial.get("page_name", "")
    if page_name in ("root", "overview", ""):
        for loc in ["tab-nav-form", "btn-nav-personal", "btn-login"]:
            r = server.click_element(loc)
            if r.get("success") and r.get("new_page"):
                break

    visited = set()
    step = 0

    while step < max_steps:
        step += 1
        page_data = server.scan_page()
        page_name = page_data.get("page_name", "unknown")
        elements = page_data.get("elements", [])

        # Skip non-form pages
        if page_name in ("root", "overview", ""):
            server.click_element("tab-nav-form")
            continue

        # Skip already-visited pages
        if page_name in visited:
            nav = server.click_element("btn-next")
            if not nav.get("new_page"):
                break
            continue
        visited.add(page_name)

        print(f"\n{'='*50}")
        print(f"Page {step}: {page_name}")
        print(f"{'='*50}")

        # Submit on review page
        if page_name == "review":
            print("Review page -- submitting!")
            result = server.submit_form()
            print(f"Submitted! Success: {result.get('success')}")
            return {
                "submission": result,
                "filled_history": filled_history,
                "plans": plans,
            }

        # Handle dynamic rows
        fillable = [e for e in elements if e.get("type") not in ("button", "text")]
        has_add_row = any(
            e.get("type") == "button" and "add-row" in (e.get("locator") or "").lower()
            for e in elements
        )
        if not fillable and has_add_row:
            new_scan = add_rows_if_needed(server, elements)
            if new_scan:
                elements = new_scan.get("elements", [])

        fillable = [e for e in elements if e.get("type") not in ("button", "text")]
        if not fillable:
            print("  No fillable fields -- skipping")
        else:
            print(f"  {len(fillable)} fillable fields")

            # ════════════════════════════════════════════════════
            # STUDENT TODO: Call think_step and fill the page
            # 1. Call think_step() to get (plan, mapping)
            # 2. Visualize the plan
            # 3. Handle _rows_needed if present
            # 4. Fill the page
            # ════════════════════════════════════════════════════

            # Step 1: Get the plan and mapping
            plan, mapping = ...  # <-- FILL IN: call think_step()

            # Step 2: Visualize
            if plan:
                plans.append({"page": page_name, "plan": plan})
                visualize_plan(page_name, plan, catalog)

            # Step 3: Handle _rows_needed
            rows_needed = int(mapping.pop("_rows_needed", 0))
            if rows_needed > 0:
                new_scan = add_rows_if_needed(server, elements, rows_needed)
                if new_scan:
                    elements = new_scan.get("elements", [])
                    _, mapping = think_step(
                        server, llm, page_name, elements,
                        catalog, all_guides, filled_history, SECTION_PROMPTS,
                    )
                    mapping.pop("_rows_needed", None)

            # Step 4: Fill the page
            if mapping:
                ...  # <-- FILL IN: call fill_page()
            else:
                print("  LLM returned empty mapping -- nothing to fill")

        # Navigate to next page
        nav_result = server.click_element("btn-next")
        if nav_result.get("success") and nav_result.get("new_page"):
            print(f"  -> Navigated to: {nav_result['new_page']}")
        else:
            for nav_target in ["tab-nav-form", "nav-personal", "nav-income",
                               "nav-deductions", "nav-wealth",
                               "nav-attachments", "nav-review"]:
                r = server.click_element(nav_target)
                if r.get("success") and r.get("new_page"):
                    print(f"  -> Sidebar jump to: {r['new_page']}")
                    break
            else:
                print("  Cannot navigate further -- stopping")
                break

    print(f"\nMax steps ({max_steps}) reached without submitting.")
    return {"submission": None, "filled_history": filled_history, "plans": plans}

In [ ]:
# ════════════════════════════════════════════════════════════════
# THE BIG CELL — Display UI + Init Server + Run Agent
# ════════════════════════════════════════════════════════════════
# This cell MUST be a single cell because Colab's eval_js()
# only works within the same output cell where HTML was rendered.
# ════════════════════════════════════════════════════════════════

# -- Change PERSONA here! --
PERSONA = "anna_meier"
# Available: 'anna_meier' (Easy), 'marco_laura_bernasconi' (Medium),
#            'priya_chakraborty' (Hard), 'thomas_elisabeth_widmer' (Very Hard),
#            'yuki_tanaka' (Expert)

# 1. Render the UI
from IPython.display import display, HTML
import os, json, datetime
from google.colab.output import eval_js
import time

BUILD_DIR = f"{REPO}/EnvDemoV1/dist"
with open(os.path.join(BUILD_DIR, "index.html")) as f:
    html_content = f.read()

html_content = html_content.replace('"\/assets/', f'"http://localhost:{PORT}/assets/')
html_content = html_content.replace('"./assets/', f'"http://localhost:{PORT}/assets/')
html_content = html_content.replace('"/assets/', f'"http://localhost:{PORT}/assets/')

display(HTML(html_content))
print(f"UI rendered from http://localhost:{PORT}")

time.sleep(1)
eval_js(f'window.TaxPortal.setPersona("{PERSONA}")')

# 2. Init MCP Server with ColabBridge
from mcp_server.server import MCPServer
from mcp_server.bridges.colab import ColabBridge

server = MCPServer(
    persona_folder=f"{REPO}/personas/{PERSONA}",
    guides_folder=f"{REPO}/guides",
    bridge=ColabBridge(),
)
print(f"Server ready for: {PERSONA}")

# 3. Run the agent
print(f"\nStarting agent for {PERSONA}...\n")
agent_result = think_loop(server, llm, max_steps=100)
print("\nDone!")

# 4. Save outputs
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = f"/content/agent_output_{PERSONA}_{timestamp}"
os.makedirs(out_dir, exist_ok=True)

with open(f"{out_dir}/filled_history.json", "w") as f:
    json.dump(agent_result.get("filled_history", {}), f, indent=2, ensure_ascii=False)
with open(f"{out_dir}/plans.json", "w") as f:
    json.dump(agent_result.get("plans", []), f, indent=2, ensure_ascii=False, default=str)
with open(f"{out_dir}/submission.json", "w") as f:
    json.dump(agent_result.get("submission", {}), f, indent=2, ensure_ascii=False, default=str)

print(f"Output saved to: {out_dir}")

In [ ]:
#@title Scoring — Compare against ground truth { display-mode: "form" }
import json
from pathlib import Path

# Load ground truth
gt_path = Path(f"{REPO}/personas/{PERSONA}/ground_truth.json")
if not gt_path.exists():
    print(f"No ground_truth.json found for {PERSONA}")
else:
    with open(gt_path) as f:
        ground_truth = json.load(f)

    # Load submission
    submission = agent_result.get("submission", {})
    if isinstance(submission, dict):
        submission_data = submission.get("submission_json", submission)
    else:
        submission_data = {}

    # Compare
    correct = 0
    wrong = 0
    missing = 0
    total = 0

    def compare_fields(gt, sub, path=""):
        nonlocal correct, wrong, missing, total
        if not isinstance(gt, dict):
            return
        for key, gt_val in gt.items():
            if key.startswith("_"):
                continue
            full_key = f"{path}.{key}" if path else key

            if isinstance(gt_val, dict):
                compare_fields(gt_val, sub.get(key, {}) if isinstance(sub, dict) else {}, full_key)
            else:
                total += 1
                sub_val = sub.get(key) if isinstance(sub, dict) else None
                if sub_val is None or sub_val == "":
                    missing += 1
                    print(f"  MISSING: {full_key} (expected: {gt_val})")
                elif str(sub_val).strip().lower() == str(gt_val).strip().lower():
                    correct += 1
                else:
                    wrong += 1
                    print(f"  WRONG: {full_key} = {sub_val} (expected: {gt_val})")

    compare_fields(ground_truth, submission_data)

    score = (correct / total * 100) if total > 0 else 0
    print(f"\nScore: {correct}/{total} ({score:.1f}%)")
    print(f"  Correct: {correct}, Wrong: {wrong}, Missing: {missing}")

## 8. Wrap-Up

### What you built

An AI agent that autonomously files a Swiss tax return using a **Plan-then-Execute** architecture:

1. **Plan phase**: The LLM sees document *names*, not content, and decides what to read
2. **Selective retrieval**: Only the documents the plan requested are loaded
3. **Execute phase**: The LLM fills fields using only the relevant documents
4. **Visualization**: Every plan is rendered as a visual card showing the agent's reasoning

### Key insights

- **Selective retrieval matters**: Sending only relevant documents reduces noise and keeps context focused. This is especially important for local LLMs with smaller context windows.
- **Planning makes agents transparent**: By splitting reasoning into plan + execute, you can see *why* the agent made each decision — not just what it did.
- **The agent doesn't see the whole form at once**: It works page by page, building up context through `filled_history`.

### Extensions to explore
- Add a **validation loop**: after filling a page, scan for validation errors and re-ask the LLM
- Use `ask_user` to handle ambiguous fields (the NPC taxpayer has verbal knowledge not in documents)
- Try different local models and compare accuracy vs speed
- Build a **confidence score** into the plan — if the LLM is uncertain, flag it for human review

### References
- [OLLAMA Documentation](https://ollama.com/)
- [OpenAI Chat Completions API](https://platform.openai.com/docs/api-reference/chat)
- [Swiss Tax System Overview](https://www.estv.admin.ch/)
- [MCP Protocol Specification](https://modelcontextprotocol.io/)